In [1]:
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from datasets.dataset_dict import DatasetDict, IterableDatasetDict
from datasets.iterable_dataset import IterableDataset
 
dataset_id = "argilla/synthetic-domain-text-classification"
 
train_dataset = load_dataset(dataset_id, split='train')

split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset['train'][0]


{'text': 'In recent years, scientists have made significant strides in understanding the complex dynamics of black holes and their impact on the fabric of space-time. These massive celestial entities are formed by the gravitational collapse of stellar cores following supernovae events. The intense gravity around a black hole distorts time and space, creating phenomena such as event horizons, where not even light can escape its grasp. Recent research has also unveiled the presence of supermassive black holes at the centers of most galaxies, including our own Milky Way. These discoveries have profound implications for astrophysics and cosmology, reshaping our understanding of the universe.',
 'label': 12}

In [2]:
from transformers import AutoTokenizer
 
model_id = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
 
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=512, return_tensors="pt")
 
if "label" in split_dataset["train"].features.keys():
    split_dataset =  split_dataset.rename_column("label", "labels") 
tokenized_dataset = split_dataset.map(tokenize, batched=True, remove_columns=["text"])
 
tokenized_dataset["train"].features.keys()


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [3]:
from transformers import AutoModelForSequenceClassification

model_id = "answerdotai/ModernBERT-base"
 
labels = tokenized_dataset["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label
 
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"f1": f1_score(labels, predictions, average="weighted")}


In [5]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="ModernBERT-domain-classifier",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True,
    optim="adamw_torch_fused",
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.569500,0.472544,0.863142
2,0.315460,0.456787,0.885556
3,0.005764,0.421135,0.897154
4,0.017703,0.456284,0.895123
5,0.000242,0.442280,0.893244


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=565, training_loss=0.3429375272409051, metrics={'train_runtime': 189.5377, 'train_samples_per_second': 23.742, 'train_steps_per_second': 2.981, 'total_flos': 1120294678932000.0, 'train_loss': 0.3429375272409051, 'epoch': 5.0})

In [3]:
from transformers import pipeline
clf = pipeline("text-classification", model="ModernBERT-domain-classifier-save", device=0)

print(clf("The team won the championship after a dramatic final match."))
print(clf("Bitcoin price dropped 10% after the new regulations were announced."))

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[{'label': 'sports', 'score': 0.9992889165878296}]
[{'label': 'finance', 'score': 0.6938598155975342}]
